### ЗАДАЧА: Пакетная загрузка отгрузок (try/except + custom exceptions)

Из внешней логистической системы приходят строки с отгрузками.
Нужно безопасно распарсить данные, отделить валидные записи от ошибок
и посчитать несколько итоговых метрик.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `ShipmentError`
   - `RowFormatError`
   - `WeightError`
   - `PriorityError`
   - `RegionError`.

2. Функцию `parse_shipment(row)`:
   - формат строки: `shipment_id,client,weight,priority,region`
   - `weight` должен быть числом и `> 0`
   - допустимые приоритеты: `standard`, `express`, `vip`
   - допустимые регионы: `RU`, `KZ`, `BY`
   - при ошибке конвертации веса использовать `raise ... from ...`.

3. Функцию `load_shipments(rows)`:
   - вернуть `(shipments, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных отгрузок,
   - ошибки по типам,
   - суммарный вес только для `express` и `vip`,
   - клиента-лидера по суммарному весу среди валидных записей.

In [ ]:
rows = [
    'S-100,Acme,12.5,express,RU',
    'S-101,Beta,0,standard,RU',
    'S-102,Acme,abc,vip,KZ',
    'S-103,Delta,8.5,urgent,BY',
    'S-104,Gamma,15,vip,UZ',
    'S-105,Acme,4.0,standard,KZ',
    'S-106,Beta,9.5,express,BY',
]


class ShipmentError(Exception):
    pass


class RowFormatError(ShipmentError):
    pass


class WeightError(ShipmentError):
    pass


class PriorityError(ShipmentError):
    pass


class RegionError(ShipmentError):
    pass


def parse_shipment(row):
    # TODO: распарсить строку и провалидировать weight, priority, region
    parts = row.split(",")
    if len(parts) != 5:
        raise RowFormatError ("Неверный формат строки")
    shipment_id, client, weight, priority, region = parts

    try:
        weight = float(weight)
    except ValueError as e:
        raise WeightError ("Вес должен быть числом")
    if weight <= 0:
        raise WeightError ("Вес должен быть положительным числом")
    
    acceptable_priorities = {"standard", "express", "vip"}
    if priority not in acceptable_priorities:
        raise PriorityError("Недопустимый статус")

    acceptable_region = {"RU", "KZ", "BY"}
    if region not in acceptable_region:
        raise RegionError("Недопустимый регион")    
    # TODO: при ошибке конвертации weight использовать raise ... from ...

    return {
        'shipment_id': shipment_id,
        'client': client,
        'weight': weight,
        'priority': priority,
        'region': region
    }


def load_shipments(rows):
    shipments = []
    errors = []

    for row in rows:
        try:
            shipment = parse_shipment(row)
            shipments.append(shipment)
        except ShipmentError as e:
            errors.append((row, type(e).__name__, e))
    return shipments, errors
    # TODO: вернуть (shipments, errors)

# TODO: вызвать load_shipments(rows)
shipments, errors = load_shipments(rows)

# TODO: вывести число валидных отгрузок и число ошибок
print(f"Количество валидных отгрузок: {len(shipments)}")
print(f"Количество ошибок: {len(errors)}")

# TODO: вывести ошибки по типам
if errors:
    print("\nДетализация ошибок:")
    error_counts = {}
    for _, error_type, _ in errors:
        error_counts[error_type] = error_counts.get(error_type, 0) + 1
    for error_type, count in error_counts.items():
        print(f"{error_type}: {count}")
    print("\nПолный список ошибок:")
    for row, error_type, message in errors:
        print(f"Строка: '{row}' | Ошибка: {error_type} | Сообщение: {message}")
else:
    print("Ошибок не обнаружено.")

# TODO: посчитать premium_weight только для express/vip
premium_weight = 0
for shipment in shipments:
    if shipment['priority'] == "express" or shipment['priority'] == 'vip':
        premium_weight += shipment['weight']
print(f"\nСуммарный вес для \"express/vip\": {premium_weight}")

# TODO: найти клиента-лидера по суммарному весу
product_weight = {}
for shipment in shipments:
    client = shipment['client']
    weight = shipment['weight']
    if client in product_weight:
        product_weight[client] += weight
    else:
        product_weight[client]  = weight
leader_client = max(product_weight, key = product_weight.get)
leader_weight = product_weight[leader_client]
print(f"Клиент-лидер по суммарному весу: {leader_client} - {leader_weight}")



